In [13]:
import torch, torchvision
from pathlib import Path
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
print("ALL MODULE IMPORTED")

ALL MODULE IMPORTED


In [14]:
dataset=Path("dataset")
test_path=dataset/"test"
train_path=dataset/"train"

transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
print("DATASET TRANSFORMATION")

DATASET TRANSFORMATION


In [15]:
train_data=ImageFolder(train_path,
                       transform=train_transform)
test_data=ImageFolder(test_path,
                      transform=transform)
print("IMAGE FOLDER LOADED")
train_loader=DataLoader(train_data,
                        batch_size=32,
                        shuffle=True)
test_loader=DataLoader(test_data,
                       batch_size=32,
                       shuffle=False)
print("DATASET LOADER SETUP")

IMAGE FOLDER LOADED
DATASET LOADER SETUP


In [29]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
weight=ResNet50_Weights
model=resnet50(weights=weight.DEFAULT)
output_features=int(input("ENTER NUMBER OF OUTPUT"))
model.fc=nn.Linear(2048, output_features)
criterion=nn.CrossEntropyLoss()
print("MODEL LOADED")

ENTER NUMBER OF OUTPUT 2


MODEL LOADED


In [30]:
optimizer=torch.optim.Adam(
    model.parameters(),
    lr=0.001)
print("OPTIMIZER SET")

OPTIMIZER SET


In [31]:
import copy
num_epoch=15
best_loss=float("inf")
best_model=None
for epoch in range(num_epoch):
 train_epoch_loss=0
 model.train()
 for inputs, labels in train_loader:
     optimizer.zero_grad()
     output=model(inputs)
     loss=criterion(output,labels)
     loss.backward()
     optimizer.step()
     train_epoch_loss+=loss.item()
 epoch_loss=train_epoch_loss/len(train_loader)
 print(f"EPOCH {epoch+1}")
 if  epoch_loss<best_loss:
     best_loss=epoch_loss
     best_model=copy.deepcopy(model.state_dict())
 print(f"TRAINING LOSS: {epoch_loss} LEAST LOSS: {best_loss}")
print("TRAINING ENDED")
model.load_state_dict(best_model)

EPOCH 1
TRAINING LOSS: 0.5793893337249756 LEAST LOSS: 0.5793893337249756
EPOCH 2
TRAINING LOSS: 0.23234374076128006 LEAST LOSS: 0.23234374076128006
EPOCH 3
TRAINING LOSS: 0.05001665744930506 LEAST LOSS: 0.05001665744930506
EPOCH 4
TRAINING LOSS: 0.07444717548787594 LEAST LOSS: 0.05001665744930506
EPOCH 5
TRAINING LOSS: 0.05337741272523999 LEAST LOSS: 0.05001665744930506
EPOCH 6
TRAINING LOSS: 0.06418010592460632 LEAST LOSS: 0.05001665744930506
EPOCH 7
TRAINING LOSS: 0.06277717463672161 LEAST LOSS: 0.05001665744930506
EPOCH 8
TRAINING LOSS: 0.006138340570032597 LEAST LOSS: 0.006138340570032597
EPOCH 9
TRAINING LOSS: 0.3015918005257845 LEAST LOSS: 0.006138340570032597
EPOCH 10
TRAINING LOSS: 0.005822399165481329 LEAST LOSS: 0.005822399165481329
EPOCH 11
TRAINING LOSS: 0.04605904594063759 LEAST LOSS: 0.005822399165481329
EPOCH 12
TRAINING LOSS: 0.016564338002353907 LEAST LOSS: 0.005822399165481329
EPOCH 13
TRAINING LOSS: 0.026735189370810986 LEAST LOSS: 0.005822399165481329
EPOCH 14
TRAIN

<All keys matched successfully>

In [33]:
model.fc=nn.Identity()
model.eval()
def featureExtracter(model,loader):
 image_feature=[]
 image_label=[] 
 with torch.no_grad():
    for inputs, labels in loader:
        output=model(inputs)
        output=nn.functional.normalize(
             output,
             p=2,
             dim=1
             )
        image_feature.append(output)
        image_label.append(labels)
 return (torch.cat(image_feature),
         torch.cat(image_label))
print("FEATURE EXTRACTOR")

FEATURE EXTRACTOR


In [39]:
features, label=featureExtracter(model, train_loader)
def prototypeCalculator(features, label, number_class):
    prototype=[]
    for c in range(number_class):
        class_feature=features[label==c]
        class_prototype=class_feature.mean(dim=0)
        prototype.append(class_prototype)
    prototype=torch.stack(prototype)
    prototype = nn.functional.normalize(
    prototype,
    p=2,
    dim=1,
    eps=1e-12
    )
    return prototype
print("PROTOTYPE CALCULATOR")

PROTOTYPE CALCULATOR


In [41]:
prototype=prototypeCalculator(features,label,output_features)
distance_from_prototype=torch.cdist(features,prototype)

In [43]:
# ============================================================
# 8. FIND THE CLOSEST KNOWN CLASS
# ============================================================
min_dists, _ = distance_from_prototype.min(
    dim=1
)
# ============================================================
# 9. FIND THE OSR THRESHOLD
# ============================================================
threshold = torch.quantile(
    min_dists,
    0.95
).item()

print(
    f"OSR Threshold: {threshold:.4f}"
)

OSR Threshold: 0.7012


In [45]:
# ============================================================
# 10. EXTRACT FEATURES FROM TEST DATA
# ============================================================
test_features, test_labels = featureExtracter(
    model,
    test_loader
)
test_distances = torch.cdist(
    test_features,
    prototype
)
# ============================================================
# 12. FIND CLOSEST CLASS FOR EACH TEST IMAGE
# ============================================================
min_test_dists, predictions = test_distances.min(
    dim=1
)
# ============================================================
# 13. REJECT IMAGES THAT ARE TOO FAR
# ============================================================
predictions[
    min_test_dists > threshold
] = -1

print("\nRESULTS")
print("--------------------")

print(
    "Predictions:",
    predictions
)

print(
    "Actual:     ",
    test_labels
)



RESULTS
--------------------
Predictions: tensor([ 0,  0, -1,  0,  0,  0,  0, -1, -1, -1,  1,  1,  1,  1, -1,  1,  1])
Actual:      tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2])


In [47]:
print("TRAIN:")
print(train_data.classes)
print(train_data.class_to_idx)

print("\nTEST:")
print(test_data.classes)
print(test_data.class_to_idx)

TRAIN:
['red', 'white']
{'red': 0, 'white': 1}

TEST:
['red', 'unknown', 'white']
{'red': 0, 'unknown': 1, 'white': 2}
